# Experiment 2 — EDA of the static CART inputs

This notebook inspects the **exact eight patient-level features passed to CART** in `experiment_2_cart_baseline.py`. It calls the script's preprocessing functions rather than duplicating aggregation logic. Run all cells manually from a Python kernel with the existing project dependencies.

The outcome is **FA**. `GI` and `Vomiting` are clinical predictors. Each event feature becomes one full-history sum: an event count. Every observed event has value 1; a mean of existing event rows would therefore be uninformative. An absent event becomes zero recorded occurrences. For `FishIntroduction`, zero means **no recorded introduction**, not proof that the patient never ate fish.

This notebook does not fit a model, choose hyperparameters, or write experiment outputs. Feature distributions and associations with FA use the original **training patients only**. Test counts and label prevalence are shown only to describe the saved split. IDs, blocks, documentation status, dates, private risk variables and labels are not model columns.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

repository_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "experiments" / "experiment_2_cart_baseline.py").is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError("Start Jupyter inside TemporalDecisionTree_clean.")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from experiments import experiment_2_cart_baseline as cart

# Importing the runner selects Agg; enable inline plotting after that import.
from IPython import get_ipython
active_shell = get_ipython()
if active_shell is not None:
    active_shell.run_line_magic("matplotlib", "inline")
pd.set_option("display.max_columns", 20)
recorded_run = cart.RECORDED_RUN
print("Preprocessing implementation:", cart.__file__)
print("Recorded dataset/run:", recorded_run)

## 1. Load and validate the exact CART input matrix

`prepare_inputs()` validates the copied dataset manifest and confirms that the original cohort, block split and FA labels match the saved Dynamic ON predictions. It builds the same matrix used by the CART script. Reading these records does not retrain TAT or CART.

In [ ]:
X, patient_info, train_ids, test_ids, _reference, configuration = (
    cart.prepare_inputs(recorded_run)
)
X_train = X.loc[train_ids]
X_test = X.loc[test_ids]
y_train = patient_info.loc[train_ids, "FA"]
y_test = patient_info.loc[test_ids, "FA"]

split_summary = pd.DataFrame([
    {
        "partition": name,
        "patients": len(ids),
        "blocks": patient_info.loc[ids, "block"].nunique(),
        "features": X.shape[1],
        "FA_positive_prevalence": patient_info.loc[ids, "FA"].mean(),
    }
    for name, ids in (("train", train_ids), ("test", test_ids))
])
display(split_summary)

## 2. Original event rows before aggregation

Raw timestamps are displayed here for context only. They are excluded before aggregation. Only `patient_id`, `feature_name` and the unit-valued `value` contribute to event counts; patient IDs become the matrix index, not a model feature. No private patient-level risk columns are displayed.

In [ ]:
longitudinal, _patients, _validation = cart.load_dataset(recorded_run)
clinical = longitudinal.loc[
    longitudinal["feature_name"].isin(cart.EVENT_FEATURES)
]
print(f"Original longitudinal rows: {len(longitudinal):,}")
display(longitudinal.head(10))
display(
    clinical.groupby("feature_name").agg(
        event_rows=("value", "size"),
        patients_with_recorded_event=("patient_id", "nunique"),
        distinct_values=("value", "nunique"),
    )
)
assert pd.to_numeric(clinical["value"]).eq(1).all()

## 3. What CART receives

Each row is one patient; each column is a full-history clinical event count. There are no labels, dates or anchor variables in these tables. Columns are identical in train and test.

In [ ]:
display(X_train.head(10))
display(X_train.head(5).T.rename_axis("CART_feature"))
display(X_train.describe().T)

## 4. Trace raw observations to feature counts

Inspect one training patient from each available documentation group. Independently count their raw event rows and compare them to all eight CART values. Documentation status is used to select illustrative examples only; it is not added to `X`.

In [ ]:
examples = patient_info.loc[train_ids].drop_duplicates("documented").index
for patient_id in examples:
    rows = clinical.loc[clinical["patient_id"].eq(patient_id)]
    raw_counts = rows["feature_name"].value_counts().reindex(
        cart.EVENT_FEATURES, fill_value=0
    )
    model_counts = X.loc[
        patient_id, [f"{feature}__sum" for feature in cart.EVENT_FEATURES]
    ]
    np.testing.assert_array_equal(raw_counts.to_numpy(), model_counts.to_numpy())
    print("Patient:", patient_id,
          "recorded fish introduction:", patient_info.loc[patient_id, "documented"])
    display(pd.DataFrame({
        "clinical_feature": cart.EVENT_FEATURES,
        "raw_occurrence_rows": raw_counts.to_numpy(),
        "CART_input_value": model_counts.to_numpy(),
    }))

## 5. Missing values, zeros and constant columns — training only

A zero is a valid recorded event count, not a missing numeric measurement. Full-history aggregation can remove distinctions between histories with different timing. Constant columns provide no discrimination; this EDA does not remove or select features.

In [ ]:
quality = pd.DataFrame({
    "dtype": X_train.dtypes.astype(str),
    "missing_values": X_train.isna().sum(),
    "zero_count": X_train.eq(0).sum(),
    "zero_fraction": X_train.eq(0).mean(),
    "unique_values": X_train.nunique(),
    "minimum": X_train.min(),
    "maximum": X_train.max(),
}).rename_axis("CART_feature")
display(quality)
print("Training-constant columns:", quality.index[quality.unique_values.le(1)].tolist())

## 6. Event-count distributions — training only

In [ ]:
figure, axes = plt.subplots(2, 4, figsize=(16, 7))
for axis, column in zip(axes.ravel(), X_train.columns):
    values = X_train[column]
    bins = np.arange(int(values.max()) + 2) - 0.5
    axis.hist(values, bins=bins)
    axis.set(title=column, xlabel="Full-history event count", ylabel="Patients")
figure.suptitle("CART input distributions — original training patients")
figure.tight_layout()
plt.show()
plt.close(figure)

## 7. Correlations between CART inputs — training only

Correlations summarize co-occurrence patterns, not causal effects. Undefined correlations for constant columns remain missing.

In [ ]:
correlations = X_train.corr()
display(correlations)
figure, axis = plt.subplots(figsize=(9, 8))
image = axis.imshow(np.ma.masked_invalid(correlations.to_numpy()), vmin=-1, vmax=1)
axis.set_xticks(range(len(correlations)))
axis.set_yticks(range(len(correlations)))
axis.set_xticklabels(correlations.columns, rotation=90)
axis.set_yticklabels(correlations.index)
axis.set_title("Training feature correlations")
figure.colorbar(image, ax=axis, label="Pearson correlation")
figure.tight_layout()
plt.show()
plt.close(figure)

## 8. Descriptive associations with FA — training only

Labels are joined into a separate **EDA table**. This table is never a model input. Compare mean counts by FA class and inspect their overlap; a difference does not establish causation or validation performance.

In [ ]:
training_eda = X_train.join(y_train.rename("FA_label"))
display(training_eda.groupby("FA_label")[X_train.columns.tolist()].mean().T)
figure, axes = plt.subplots(2, 4, figsize=(16, 7))
for axis, column in zip(axes.ravel(), X_train.columns):
    groups = [training_eda.loc[training_eda.FA_label.eq(label), column]
              for label in (0, 1)]
    axis.boxplot(groups)
    axis.set_xticks([1, 2])
    axis.set_xticklabels(["FA=0", "FA=1"])
    axis.set(title=column, ylabel="Full-history event count")
figure.suptitle("Training event counts by FA label")
figure.tight_layout()
plt.show()
plt.close(figure)

## 9. The static fish-record signal — training only

`FishIntroduction__sum` is 0 or 1 in this benchmark. The existing clinical record therefore exposes whether an introduction was documented, even after dates are removed. Inspecting class prevalence in these groups helps explain why a static model can perform above chance. This is a descriptive check; no decision threshold or model is selected here.

In [ ]:
assert X_train["FishIntroduction__sum"].isin([0, 1]).all()
fish_record_eda = pd.DataFrame({
    "has_fish_record": X_train["FishIntroduction__sum"].gt(0),
    "FA_label": y_train,
})
display(fish_record_eda.groupby("has_fish_record").agg(
    patients=("FA_label", "size"),
    FA_positive_prevalence=("FA_label", "mean"),
))
np.testing.assert_array_equal(
    fish_record_eda["has_fish_record"].to_numpy(),
    patient_info.loc[train_ids, "documented"].to_numpy(dtype=bool),
)

## 10. Identical static rows with different labels — training only

Matched histories can have identical event counts despite different event timing. A static tree must assign the same score to identical feature rows. Mixed labels can also arise from the benchmark's stochastic outcomes. This table describes the available representation; it is not a performance estimate or a reason to drop patients.

In [ ]:
identical_rows = training_eda.groupby(X_train.columns.tolist()).agg(
    patients=("FA_label", "size"),
    label_classes=("FA_label", "nunique"),
    FA_positive_prevalence=("FA_label", "mean"),
)
mixed = identical_rows.loc[identical_rows.label_classes.eq(2)]
print("Distinct static feature rows:", len(identical_rows))
print("Feature groups containing both FA labels:", len(mixed))
print("Training patients in these mixed-label groups:", int(mixed.patients.sum()))
display(mixed.sort_values("patients", ascending=False).head(10))

## 11. Scientific validation checks

Verify patient/label alignment, nonnegative integer counts, the feature allowlist and block separation. Finally, remove all timestamps and anchor values in a temporary in-memory copy and shuffle the rows: CART inputs must remain exactly identical. The saved dataset is never changed.

In [ ]:
expected_columns = [f"{feature}__sum" for feature in cart.EVENT_FEATURES]
assert X.columns.tolist() == expected_columns
assert X.index.is_unique and set(X.index) == set(patient_info.index)
assert X_train.columns.equals(X_test.columns)
assert set(train_ids).isdisjoint(test_ids)
assert set(train_ids) | set(test_ids) == set(X.index)
assert set(patient_info.loc[train_ids, "block"]).isdisjoint(
    patient_info.loc[test_ids, "block"]
)
assert np.isfinite(X.to_numpy()).all()
assert X.ge(0).all().all()
np.testing.assert_array_equal(X.to_numpy(), np.floor(X.to_numpy()))
assert y_train.index.equals(X_train.index)
assert y_test.index.equals(X_test.index)
assert y_train.isin([0, 1]).all() and y_test.isin([0, 1]).all()

without_timing = longitudinal.copy()
without_timing["time"] = pd.NaT
without_timing.loc[
    without_timing.feature_name.eq("Birth Date Anchor"), "value"
] = "ignored anchor"
shuffled = without_timing.sample(frac=1, random_state=42)
pd.testing.assert_frame_equal(X, cart.aggregate_events(shuffled, patient_info.index))
print("PASS: exact clinical-only CART columns; labels and partitions aligned.")
print("PASS: blocks isolated; dates, anchors and event order do not affect CART inputs.")
print("No model fitted and no experiment outputs written.")